## Accuracy Evaluation for AdaptViT

Evaluates top-1 accuracy of pruned ViT models on the Oxford-IIIT Pets
test set. Two evaluation modes are covered:

  1. FP32  — loads pre-saved pruned models at full float32 precision, no masks
  2. INT8  — dynamic INT8 backbone + C-equivalent INT16 classifier,
             precision-matched to the TVM-generated C code running on RISC-V

The INT8 evaluation has two sub-modes: sample inference on two fixed test
images (useful for cross-checking against C output) and full test set
accuracy reporting.

Run this notebook after linear probing has saved all head_mlp.pt files.
The same fixed test split (seed 2026) used during linear probing is
reproduced here so results are directly comparable.

Author: Vishnu PS

## Environment Setup

In [ ]:
!git clone https://github.com/WalterSimoncini/SnapViT.git 2>/dev/null || echo "Already cloned"
%cd SnapViT
!pip install -r requirements.txt
!pip install numpy==1.26.4
!pip install torch torchvision
!pip install timm

In [ ]:
import os
os.environ['TIMM_FUSED_ATTN'] = '0'
import sys
from google.colab import drive

sys.path.insert(0, '/content/SnapViT')
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

## Dataset Setup

In [ ]:
import os
import re

LOCAL_IMAGES = "/content/local_pets"
RAW_DIR      = "/tmp/pets_raw"
TAR_PATH     = "/tmp/pets.tar.gz"

os.makedirs(LOCAL_IMAGES, exist_ok=True)

existing_breeds = [d for d in os.listdir(LOCAL_IMAGES)
                   if os.path.isdir(os.path.join(LOCAL_IMAGES, d))]

if len(existing_breeds) >= 37:
    print(f"Already organised this session ({len(existing_breeds)} breeds) — skipping")
else:
    print("Downloading Oxford-IIIT Pets images (~800 MB)...")
    !wget -q --show-progress -O {TAR_PATH} \
        https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz

    print("Extracting...")
    os.makedirs(RAW_DIR, exist_ok=True)
    !tar -xf {TAR_PATH} -C {RAW_DIR} --strip-components=1

    print("Organising into breed subfolders...")
    pattern = re.compile(r'^(.+)_\d+\.jpg$', re.IGNORECASE)
    for fname in os.listdir(RAW_DIR):
        m = pattern.match(fname)
        if not m:
            continue
        breed = m.group(1)
        breed_dir = os.path.join(LOCAL_IMAGES, breed)
        os.makedirs(breed_dir, exist_ok=True)
        src = os.path.join(RAW_DIR, fname)
        dst = os.path.join(breed_dir, fname)
        if not os.path.exists(dst):
            os.rename(src, dst)

    breeds       = [d for d in os.listdir(LOCAL_IMAGES)
                    if os.path.isdir(os.path.join(LOCAL_IMAGES, d))]
    total_images = sum(len(os.listdir(os.path.join(LOCAL_IMAGES, b))) for b in breeds)
    print(f"\nDATASET READY!")
    print(f"Breeds   : {len(breeds)}")
    print(f"Images   : {total_images}")
    print(f"Location : {LOCAL_IMAGES}")

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
import random
import sys
import os
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

IMAGES_PATH = LOCAL_IMAGES

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


dataset = datasets.ImageFolder(IMAGES_PATH, transform=transform)

indices = list(range(len(dataset)))
random.seed(2026)
random.shuffle(indices)

test_size = int(0.2 * len(dataset))
trainval_indices = indices[:-test_size]
test_indices     = indices[-test_size:]

trainval_dataset = Subset(dataset, trainval_indices)
test_dataset     = Subset(dataset, test_indices)

train_size = int(0.8 * len(trainval_dataset))
val_size   = len(trainval_dataset) - train_size

train_set, val_set = random_split(
    trainval_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(val_set,   batch_size=128, shuffle=False,
                          num_workers=8, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False,
                          num_workers=8, pin_memory=True, persistent_workers=True)

print(f"Total images     : {len(dataset)}")
print(f"Train            : {len(train_set)}")
print(f"Validation       : {len(val_set)}")
print(f"Test (held-out)  : {len(test_dataset)}")
print(f"Classes          : {len(dataset.classes)}")

CLASS_NAMES = dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

In [ ]:
import os
import torch
import torch.nn as nn
import sys
from tqdm import tqdm

sys.path.insert(0, '/content/SnapViT')

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_PATH     = '/content/drive/MyDrive/PhD_Works/SnapViT_Quantized'
DATASET_PATH  = "/content/local_pets"

# Discover models automatically
MODELS_ROOT = f'{BASE_PATH}/model_outputs'
MODEL_NAMES = sorted([
    d for d in os.listdir(MODELS_ROOT)
    if os.path.isdir(os.path.join(MODELS_ROOT, d))
])

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=16,      # smaller batch size
    shuffle=False,
    num_workers=0,      # very important
    pin_memory=False
)
print("Test loader recreated safely.")

## FP32 accuracy (extended SnapViT, per-group pruning)
Loads each pruned model object (full_pruned_model.pt) saved by the pruning
notebook and evaluates it at full float32 precision. No masks are applied
here, the pruned weights are baked into the model object itself, so this
reflects the accuracy of the pruning.

In [ ]:
import os
import torch
import torch.nn as nn
import timm
from tqdm import tqdm

print("="*110)
print("CELL Q3b: FULL TEST SET ACCURACY — FP32 ONLY")
print("="*110 + "\n")
print(f"Test set size: {len(test_dataset)} images\n")


MODEL_NAMES = ['augreg_vits16', 'augreg_vitb16', 'dino_vits16', 'dino_vitb16']

PRUNING_LEVELS = [
    (0.15, 0.0), (0.25, 0.1), (0.35, 0.2),
    (0.45, 0.3), (0.55, 0.4), (0.65, 0.5),
]

TIMM_MODEL_NAMES = {
    'augreg_vits16': 'timm/vit_small_patch16_224.augreg_in21k_ft_in1k',
    'augreg_vitb16': 'timm/vit_base_patch16_224.augreg_in21k_ft_in1k',
    'dino_vits16':   'timm/vit_small_patch16_224.dino',
    'dino_vitb16':   'timm/vit_base_patch16_224.dino',
}


all_results = {}

for MODEL_NAME in MODEL_NAMES:
    print(f"\n{'#'*100}")
    print(f"MODEL: {MODEL_NAME}")
    print(f"{'#'*100}\n")

    model_dir = os.path.join(MODELS_ROOT, MODEL_NAME)
    all_results[MODEL_NAME] = {}

    levels = []

    base_head_path = os.path.join(model_dir, 'head_mlp_base.pt')
    base_model_path = os.path.join(model_dir, 'full_pruned_model.pt')

    if os.path.exists(base_head_path):
        levels.append(('base', base_model_path if os.path.exists(base_model_path) else None, base_head_path))
    else:
        print(f" head_mlp_base.pt missing — skipping base")

    for mlp_level, head_level in PRUNING_LEVELS:
        pruning_folder = f"mlp-{mlp_level}-heads-{head_level}"
        level_dir = os.path.join(model_dir, pruning_folder)

        head_path = os.path.join(level_dir, 'head_mlp.pt')
        full_model_path = os.path.join(level_dir, 'full_pruned_model.pt')

        if os.path.exists(head_path) and os.path.exists(full_model_path):
            levels.append((pruning_folder, full_model_path, head_path))
        else:
            print(f"  [{pruning_folder}] missing files — skipping")

    for level_name, full_model_path, head_path in levels:
        print(f"→ {level_name}")

        if level_name == 'base' and full_model_path is None:

            timm_name = TIMM_MODEL_NAMES[MODEL_NAME]
            print(f"   Loading base timm model: {timm_name}")
            model = timm.create_model(timm_name, pretrained=True, num_classes=0).to(device).eval()
        else:

            print(f"   Loading full pruned model: {full_model_path}")
            model = torch.load(full_model_path, map_location=device, weights_only=False)
            model = model.to(device).eval()

        print(f"   Loading head: {head_path}")
        head_state = torch.load(head_path, map_location='cpu', weights_only=False)
        w_fp32 = head_state['weight'].detach().float()
        b_fp32 = head_state['bias'].detach().float()
        num_features = getattr(model, 'embed_dim', None)
        if num_features is None and hasattr(model, 'head'):
            num_features = getattr(model.head, 'in_features', None)

        new_head = nn.Linear(num_features, NUM_CLASSES, bias=True).to(device)
        new_head.weight.data = w_fp32.to(device)
        new_head.bias.data = b_fp32.to(device)

        if hasattr(model, 'head'):
            model.head = new_head
        else:
            model.head = new_head

        correct = total = 0
        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc=f"  FP32 {level_name}", leave=False):
                images = images.to(device)
                labels = labels.to(device)

                feats = model.forward_features(images)
                if isinstance(feats, tuple):
                    feats = feats[0]
                if feats.dim() == 3:
                    feats = feats[:, 0]

                logits = new_head(feats)

                _, predicted = logits.max(1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)

        acc_fp32 = 100. * correct / total
        all_results[MODEL_NAME][level_name] = {'fp32': acc_fp32}
        print(f"   FP32 Accuracy: {acc_fp32:.2f}%")

        del model, new_head
        torch.cuda.empty_cache()

print("\n\n" + "="*110)
print("SUMMARY: FP32 ACCURACY")
print("="*110)

for MODEL_NAME, results in all_results.items():
    print(f"\n{MODEL_NAME}:")
    print(f"  {'Level':<30s} {'FP32 Accuracy':>18s}")
    print(f"  {'-'*55}")
    for level_name, r in sorted(results.items(), key=lambda x: x[0] if x[0] != 'base' else '000'):
        print(f"  {level_name:<30s} {r['fp32']:>17.2f}%")

print("\n" + "="*110)
print("CELL Q3b COMPLETED")
print("="*110)

## Quantization setup  (run once, do not re-run)
Loads each base model from timm, applies PyTorch dynamic INT8 quantization
to all Linear and Conv2d layers, and stores the quantized backbone in
`quantized_models`. The classifier is NOT quantized here — it is handled
separately in next section, to match the C code's INT16 scheme exactly.
Quantized models must stay on CPU, PyTorch quantized ops have no CUDA backend.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.quantization as quantization
import timm

print("\n" + "="*100)
print("QUANTIZATION SETUP: Loading and Quantizing Models for Evaluation")
print("="*100 + "\n")

BASE_PATH = '/content/drive/MyDrive/PhD_Works/SnapViT_Q2'
MODELS_FOLDER = f'{BASE_PATH}/model_outputs'
DATASET_PATH = "/content/local_pets"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Evaluation Device: {device}")
print(f"Quantization will be done on CPU, then models moved to {device}\n")

TIMM_MODEL_NAMES = {
    'vit_tiny_r':    'timm/vit_tiny_r_s16_p8_224.augreg_in21k_ft_in1k',
    'augreg_vits16': 'timm/vit_small_patch16_224.augreg_in21k_ft_in1k',
    'augreg_vitb16': 'timm/vit_base_patch16_224.augreg_in21k_ft_in1k',
    'dino_vits16':   'timm/vit_small_patch16_224.dino',
    'dino_vitb16':   'timm/vit_base_patch16_224.dino',
    'deit3_vits16':  'timm/deit3_small_patch16_224.fb_in22k_ft_in1k',
    'deit3_vitb16':  'timm/deit3_base_patch16_224.fb_in22k_ft_in1k',
}

CLASS_NAMES = sorted([d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))])
NUM_CLASSES = len(CLASS_NAMES)
print(f"Number of classes: {NUM_CLASSES}\n")

print("Loading and quantizing models (Backbone: INT8)...\n")

quantized_models = {}
models_to_quantize = ['augreg_vits16', 'dino_vits16',  'dino_vitb16',  'augreg_vitb16']

for model_name in models_to_quantize:
    print(f"{'─'*80}")
    print(f"Loading and Quantizing: {model_name}")
    print(f"{'─'*80}\n")

    head_path = os.path.join(MODELS_FOLDER, model_name, "head_mlp_base.pt")

    if not os.path.exists(head_path):
        print(f" head_mlp_base.pt not found at {head_path}")
        print(f" Skipping {model_name}\n")
        continue

    try:
        print(f"  Loading base model...")
        timm_name = TIMM_MODEL_NAMES[model_name]
        model = timm.create_model(timm_name, pretrained=True, num_classes=0)
        model.head = nn.Linear(model.embed_dim, NUM_CLASSES)
        head_state = torch.load(head_path, map_location='cpu')
        model.head.load_state_dict(head_state, strict=False)
        model = model.eval().cpu()
        print(f"  ✓ Base model loaded on CPU")

        # Apply INT8 Dynamic Quantization (to backbone)
        print(f"  Applying INT8 dynamic quantization to backbone...")
        model_quantized = quantization.quantize_dynamic(
            model,
            qconfig_spec={torch.nn.Linear, torch.nn.Conv2d},
            dtype=torch.qint8,
        )

        # Verify quantization
        quant_linear = sum(1 for m in model_quantized.modules()
                          if isinstance(m, torch.nn.quantized.Linear))
        quant_conv = sum(1 for m in model_quantized.modules()
                        if isinstance(m, torch.nn.quantized.Conv2d))

        print(f"   INT8 quantization applied to backbone")
        print(f"    Quantized Linear layers: {quant_linear}")
        print(f"    Quantized Conv2d layers: {quant_conv}")
        print(f"  Model kept on CPU (quantized ops require CPU backend)")
        quantized_models[model_name] = model_quantized
        print(f"  {model_name} ready for evaluation\n")

    except Exception as e:
        print(f"  Error processing {model_name}: {e}\n")
        continue


print("="*100)
print("QUANTIZATION SETUP COMPLETE")
print("="*100 + "\n")

print("Quantized Models Ready for Evaluation:")
for model_name in quantized_models.keys():
    print(f"  {model_name}")

print(f"\nTotal Models Quantized: {len(quantized_models)}")
print(f"Models Skipped: {'vit_tiny_r (TVM incompatibility)'}")

print("\n" + "─"*100)
print("QUANTIZATION SCHEME:")
print("─"*100)

## Sample inference (quantized, C-equivalent)
Runs inference on two fixed test images across all pruning levels, using the
full precision-matched pipeline: INT8 backbone + INT16 classifier with the
exact scale constant (1/127) and softmax implementation from
the TVM-generated C code. Useful for sanity-checking that Python and C
produce the same prediction before running on RISC-V.

Pruning is reproduced at inference time: head masks zero out
Q/K/V projections for pruned heads, and MLP masks zero the FC1 outputs for
pruned neuron groups, matching the mask-check skip logic in the C runtime.

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import timm
from torchvision import transforms

print("="*110)
print("CELL Q1b: SAMPLE INFERENCE — QUANTIZED BACKBONE + C-EQUIVALENT CLASSIFIER")
print("="*110 + "\n")

BASE_PATH    = '/content/drive/MyDrive/PhD_Works/SnapViT_Quantized'
DATASET_PATH = "/content/local_pets"
MODELS_ROOT  = f'{BASE_PATH}/model_outputs'

MODEL_NAMES = ['augreg_vits16', 'dino_vits16', 'dino_vitb16', 'augreg_vitb16']

PRUNING_LEVELS = [
    (0.15, 0.0), (0.25, 0.1), (0.35, 0.2),
    (0.45, 0.3), (0.55, 0.4), (0.65, 0.5),
]

TEST_IMAGES = [
    f'{DATASET_PATH}/Russian_Blue/Russian_Blue_10.jpg',
    f'{DATASET_PATH}/British_Shorthair/British_Shorthair_89.jpg',
]

CLASS_NAMES = sorted([d for d in os.listdir(DATASET_PATH)
                      if os.path.isdir(os.path.join(DATASET_PATH, d))])
NUM_CLASSES  = len(CLASS_NAMES)

TIMM_MODEL_NAMES = {
    'vit_tiny_r':    'timm/vit_tiny_r_s16_p8_224.augreg_in21k_ft_in1k',
    'augreg_vits16': 'timm/vit_small_patch16_224.augreg_in21k_ft_in1k',
    'augreg_vitb16': 'timm/vit_base_patch16_224.augreg_in21k_ft_in1k',
    'dino_vits16':   'timm/vit_small_patch16_224.dino',
    'dino_vitb16':   'timm/vit_base_patch16_224.dino',
    'deit3_vits16':  'timm/deit3_small_patch16_224.fb_in22k_ft_in1k',
    'deit3_vitb16':  'timm/deit3_base_patch16_224.fb_in22k_ft_in1k',
}

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

C_SCALE_CONSTANT = 7.874016e-03

def softmax_like_c(logits: torch.Tensor) -> torch.Tensor:

    max_val = logits.max()
    exp_logits = torch.exp(logits - max_val)
    probs = exp_logits / exp_logits.sum()
    return probs


def apply_classifier_like_c(feats_fp32: torch.Tensor,
                             w_fp32:    torch.Tensor,
                             b_fp32:    torch.Tensor,
                             debug: bool = False) -> torch.Tensor:

    feat_min = feats_fp32.min().item()
    feat_max = feats_fp32.max().item()
    zero_point = min(feat_min, 0.0)
    max_clamped = max(feat_max, 0.0)
    feat_scale = (max_clamped - zero_point) * C_SCALE_CONSTANT

    if debug:
        print(f"    [Feature Quantization]")
        print(f"      Range: [{feat_min:.6f}, {feat_max:.6f}]")
        print(f"      Zero-point: {zero_point:.6f}")
        print(f"      Scale (C constant): {feat_scale:.10f}")
        print(f"      Scale (div by 127): {(max_clamped - zero_point) / 127.0:.10f}")
        print(f"      Precision diff:     {abs(feat_scale - (max_clamped - zero_point) / 127.0):.2e}")

    zp_normalized = zero_point / feat_scale
    neg_zp = -zp_normalized
    zp_clipped = min(neg_zp, 127.0)
    zp_offset = torch.tensor(
        round(max(zp_clipped, 0.0)),
        dtype=torch.float32
    )

    if debug:
        print(f"      ZP normalized: {zp_normalized:.6f}")
        print(f"      -ZP: {neg_zp:.6f}")
        print(f"      ZP clipped: {zp_clipped:.6f}")
        print(f"      ZP offset: {zp_offset.item():.1f}")

    quantized_fp = torch.round(feats_fp32 / feat_scale) + zp_offset
    quantized_clipped = quantized_fp.clamp(0, 255)
    quantized_uint8 = quantized_clipped.to(torch.int32)
    feats_int16 = quantized_uint8 - zp_offset.to(torch.int32)

    w_scale = w_fp32.abs().max().item() * C_SCALE_CONSTANT
    w_scale = max(w_scale, 1e-8)

    # Quantize to INT8 range (but store in INT16 for safe computation)
    w_int16 = (w_fp32 / w_scale).round().clamp(-128, 127).to(torch.int32)

    if debug:
        print(f"\n    [Weight Quantization]")
        print(f"      Scale (C constant): {w_scale:.10f}")
        print(f"      Scale (div by 127): {w_fp32.abs().max().item() / 127.0:.10f}")
        print(f"      Precision diff:     {abs(w_scale - w_fp32.abs().max().item() / 127.0):.2e}")
        print(f"      INT8 range: [{w_int16.min()}, {w_int16.max()}]")

    dot = (w_int16 * feats_int16).sum(dim=1)
    dequant_scale = feat_scale * w_scale
    logits = dot.float() * dequant_scale + b_fp32

    if debug:
        print(f"\n    [Logits]")
        print(f"      Dequant scale: {dequant_scale:.12f}")
        print(f"      Range: [{logits.min():.6f}, {logits.max():.6f}]")

    probs = softmax_like_c(logits)
    return probs.unsqueeze(0)

def masked_forward_cpu(base, x, head_masks, mlp_masks):
    num_heads  = base.blocks[0].attn.num_heads
    head_dim   = base.embed_dim // num_heads
    hidden_dim = base.blocks[0].mlp.fc1.out_features

    x = base.patch_embed(x)
    cls_token = base.cls_token.expand(x.shape[0], -1, -1)
    x = torch.cat((cls_token, x), dim=1)
    x = base.pos_drop(x + base.pos_embed)

    for i, blk in enumerate(base.blocks):
        normed = blk.norm1(x)
        B, N, C = normed.shape
        qkv = blk.attn.qkv(normed).reshape(B, N, 3, num_heads, head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        v = v * head_masks[i].view(1, num_heads, 1, 1)

        attn = (q @ k.transpose(-2, -1)) * blk.attn.scale
        attn = attn.softmax(dim=-1)
        attn = blk.attn.attn_drop(attn)

        attn_out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        attn_out = blk.attn.proj(attn_out)
        attn_out = blk.attn.proj_drop(attn_out)
        x = x + attn_out

        normed2  = blk.norm2(x)
        fc1_out  = blk.mlp.fc1(normed2)
        act_out  = blk.mlp.act(fc1_out)
        if hasattr(blk.mlp, 'drop1'):
            act_out = blk.mlp.drop1(act_out)
        act_out  = act_out * mlp_masks[i].view(1, 1, hidden_dim)
        fc2_out  = blk.mlp.fc2(act_out)
        if hasattr(blk.mlp, 'drop2'):
            fc2_out = blk.mlp.drop2(fc2_out)
        x = x + fc2_out

    x = base.norm(x)
    return x[:, 0]

for MODEL_NAME in MODEL_NAMES:
    print(f"\n{'#'*110}")
    print(f"MODEL: {MODEL_NAME}")
    print(f"{'#'*110}")

    if MODEL_NAME not in quantized_models:
        print(f"  Not in quantized_models — skipping")
        continue

    quant_backbone = quantized_models[MODEL_NAME]

    for mlp_level, head_level in PRUNING_LEVELS:
        pruning_folder = f"mlp-{mlp_level}-heads-{head_level}"
        model_folder   = os.path.join(MODELS_ROOT, MODEL_NAME, pruning_folder)
        head_path      = os.path.join(model_folder, 'head_mlp.pt')
        masks_path     = os.path.join(model_folder, 'pruning_masks.pt')

        if not all(os.path.exists(p) for p in [head_path, masks_path]):
            print(f"  [{pruning_folder}] Missing files — skipping")
            continue

        print(f"\n→ {pruning_folder}")

        head_state = torch.load(head_path, map_location='cpu')
        w_fp32 = head_state['weight'].detach().float().cpu()
        b_fp32 = head_state['bias'].detach().float().cpu()

        masks      = torch.load(masks_path, map_location='cpu')
        head_masks = masks['head_masks']
        mlp_masks  = masks['mlp_masks']

        w_scale = w_fp32.abs().max().item() * C_SCALE_CONSTANT
        print(f"   Weight scale (int16): {w_scale:.8f}")

        for img_path in TEST_IMAGES:
            if not os.path.exists(img_path):
                continue
            true_breed = os.path.basename(os.path.dirname(img_path))
            img    = Image.open(img_path).convert('RGB')
            tensor = transform(img).unsqueeze(0).cpu()

            print(f"\n   Image: {os.path.basename(img_path)} | True Breed: {true_breed}")

            with torch.no_grad():
                feats  = masked_forward_cpu(
                    quant_backbone, tensor, head_masks, mlp_masks
                )

                probs  = apply_classifier_like_c(feats, w_fp32, b_fp32, debug=False)

            probs = probs[0]
            pred  = CLASS_NAMES[probs.argmax().item()]
            conf  = probs.max().item()
            print(f"   C-equivalent (int16) → Predicted: {pred:25s}  Confidence: {conf:.4f}")
            print("      Top 5:")
            top5_p, idx5_p = torch.topk(probs, 5)
            for i in range(5):
                print(f"         {i+1}. {CLASS_NAMES[idx5_p[i].item()]:25s}  {top5_p[i].item():.6f}")

        print(f"  {pruning_folder} done")

print("\n" + "="*110)
print("CELL Q1b COMPLETED (PRECISION-MATCHED TO C CODE!)")
print("="*110)

## Full test set accuracy (quantized)
Same precision-matched quantization as Section 3a, but evaluated over the
full held-out test set. Reports both FP32 and quantized accuracy side by
side so the accuracy cost of INT8 quantization is visible per pruning level.

In [ ]:
import os
import torch
import torch.nn as nn
import timm
from tqdm import tqdm

print("="*110)
print("CELL Q3b: FULL TEST SET ACCURACY — FP32 vs QUANTIZED (PRECISION-MATCHED)")
print("="*110 + "\n")
print(f"Test set size: {len(test_dataset)} images\n")

TIMM_MODEL_NAMES = {
    'vit_tiny_r':    'timm/vit_tiny_r_s16_p8_224.augreg_in21k_ft_in1k',
    'augreg_vits16': 'timm/vit_small_patch16_224.augreg_in21k_ft_in1k',
    'augreg_vitb16': 'timm/vit_base_patch16_224.augreg_in21k_ft_in1k',
    'dino_vits16':   'timm/vit_small_patch16_224.dino',
    'dino_vitb16':   'timm/vit_base_patch16_224.dino',
    'deit3_vits16':  'timm/deit3_small_patch16_224.fb_in22k_ft_in1k',
    'deit3_vitb16':  'timm/deit3_base_patch16_224.fb_in22k_ft_in1k',
}

PRUNING_LEVELS = [
    (0.15, 0.0), (0.25, 0.1), (0.35, 0.2),
    (0.45, 0.3), (0.55, 0.4), (0.65, 0.5),
]

MODEL_NAMES = ['augreg_vits16', 'dino_vits16', 'dino_vitb16', 'augreg_vitb16']

C_SCALE_CONSTANT = 1.0 / 127.0

def softmax_like_c(logits: torch.Tensor) -> torch.Tensor:

    max_val = logits.max(dim=-1, keepdim=True).values
    exp_logits = torch.exp(logits - max_val)
    return exp_logits / exp_logits.sum(dim=-1, keepdim=True)


def masked_forward_fp32(base, x, head_layer, head_masks, mlp_masks):
    """FP32 forward with pruning masks"""
    num_heads  = base.blocks[0].attn.num_heads
    head_dim   = base.embed_dim // num_heads
    hidden_dim = base.blocks[0].mlp.fc1.out_features

    x = base.patch_embed(x)
    cls_token = base.cls_token.expand(x.shape[0], -1, -1)
    x = torch.cat((cls_token, x), dim=1)
    x = base.pos_drop(x + base.pos_embed)

    for i, blk in enumerate(base.blocks):
        normed = blk.norm1(x)
        B, N, C = normed.shape
        qkv = blk.attn.qkv(normed).reshape(B, N, 3, num_heads, head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        v = v * head_masks[i].view(1, num_heads, 1, 1)

        attn = (q @ k.transpose(-2, -1)) * blk.attn.scale
        attn = attn.softmax(dim=-1)
        attn = blk.attn.attn_drop(attn)

        attn_out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        attn_out = blk.attn.proj(attn_out)
        attn_out = blk.attn.proj_drop(attn_out)
        x = x + attn_out

        normed2  = blk.norm2(x)
        fc1_out  = blk.mlp.fc1(normed2)
        act_out  = blk.mlp.act(fc1_out)
        if hasattr(blk.mlp, 'drop1'):
            act_out = blk.mlp.drop1(act_out)
        act_out  = act_out * mlp_masks[i].view(1, 1, hidden_dim)
        fc2_out  = blk.mlp.fc2(act_out)
        if hasattr(blk.mlp, 'drop2'):
            fc2_out = blk.mlp.drop2(fc2_out)
        x = x + fc2_out

    x = base.norm(x)
    return head_layer(x[:, 0])

def masked_forward_quant(base, x, head_masks, mlp_masks):
    num_heads  = base.blocks[0].attn.num_heads
    head_dim   = base.embed_dim // num_heads
    hidden_dim = base.blocks[0].mlp.fc1.out_features

    x = base.patch_embed(x)
    cls_token = base.cls_token.expand(x.shape[0], -1, -1)
    x = torch.cat((cls_token, x), dim=1)
    x = base.pos_drop(x + base.pos_embed)

    for i, blk in enumerate(base.blocks):
        normed = blk.norm1(x)
        B, N, C = normed.shape
        qkv = blk.attn.qkv(normed).reshape(B, N, 3, num_heads, head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        v = v * head_masks[i].view(1, num_heads, 1, 1)

        attn = (q @ k.transpose(-2, -1)) * blk.attn.scale
        attn = attn.softmax(dim=-1)
        attn = blk.attn.attn_drop(attn)

        attn_out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        attn_out = blk.attn.proj(attn_out)
        attn_out = blk.attn.proj_drop(attn_out)
        x = x + attn_out

        normed2  = blk.norm2(x)
        fc1_out  = blk.mlp.fc1(normed2)
        act_out  = blk.mlp.act(fc1_out)
        if hasattr(blk.mlp, 'drop1'):
            act_out = blk.mlp.drop1(act_out)
        act_out  = act_out * mlp_masks[i].view(1, 1, hidden_dim)
        fc2_out  = blk.mlp.fc2(act_out)
        if hasattr(blk.mlp, 'drop2'):
            fc2_out = blk.mlp.drop2(fc2_out)
        x = x + fc2_out

    x = base.norm(x)
    return x[:, 0]

def apply_classifier_like_c_batch(feats_fp32: torch.Tensor,
                                   w_fp32:    torch.Tensor,
                                   b_fp32:    torch.Tensor) -> torch.Tensor:
    logits_list = []

    for i in range(feats_fp32.shape[0]):
        feat = feats_fp32[i]  # [embed_dim]

        feat_min = feat.min().item()
        feat_max = feat.max().item()
        zero_point = min(feat_min, 0.0)
        max_clamped = max(feat_max, 0.0)
        feat_scale = (max_clamped - zero_point) * C_SCALE_CONSTANT
        zp_normalized = zero_point / feat_scale
        neg_zp = -zp_normalized
        zp_clipped = min(neg_zp, 127.0)
        zp_offset = torch.tensor(round(max(zp_clipped, 0.0)), dtype=torch.float32)
        quantized_fp = torch.round(feat / feat_scale) + zp_offset
        quantized_clipped = quantized_fp.clamp(0, 255)
        quantized_uint8 = quantized_clipped.to(torch.int32)
        feats_int16 = quantized_uint8 - zp_offset.to(torch.int32)
        w_scale = w_fp32.abs().max().item() * C_SCALE_CONSTANT
        w_scale = max(w_scale, 1e-8)
        w_int16 = (w_fp32 / w_scale).round().clamp(-128, 127).to(torch.int32)
        dot = (w_int16 * feats_int16).sum(dim=1)
        logit = dot.float() * (feat_scale * w_scale) + b_fp32
        logits_list.append(logit)

    return torch.stack(logits_list, dim=0)

all_results = {}

for MODEL_NAME in MODEL_NAMES:
    print(f"\n{'#'*100}")
    print(f"MODEL: {MODEL_NAME}")
    print(f"{'#'*100}")

    if MODEL_NAME not in quantized_models:
        print(f"  Not in quantized_models — skipping")
        continue

    timm_name      = TIMM_MODEL_NAMES[MODEL_NAME]
    quant_backbone = quantized_models[MODEL_NAME]
    all_results[MODEL_NAME] = {}

    levels = []

    base_head_path = os.path.join(MODELS_ROOT, MODEL_NAME, 'head_mlp_base.pt')
    if os.path.exists(base_head_path):
        levels.append(('base', base_head_path, None))
    else:
        print(f"  head_mlp_base.pt not found — skipping base level")

    for mlp_level, head_level in PRUNING_LEVELS:
        pruning_folder = f"mlp-{mlp_level}-heads-{head_level}"
        model_folder   = os.path.join(MODELS_ROOT, MODEL_NAME, pruning_folder)
        head_path      = os.path.join(model_folder, 'head_mlp.pt')
        masks_path     = os.path.join(model_folder, 'pruning_masks.pt')

        if os.path.exists(head_path) and os.path.exists(masks_path):
              levels.append((pruning_folder, head_path, masks_path))
        else:
              print(f"  [{pruning_folder}] Missing files — skipping")

    print(f"\n  Loading fp32 backbone ({timm_name})...")
    fp32_base = timm.create_model(timm_name, pretrained=True, num_classes=0).to(device).eval()
    print(f"  FP32 backbone loaded\n")

    for level_name, head_path, masks_path in levels:
        print(f"→ {level_name}")

        head_state = torch.load(head_path, map_location='cpu')
        w_fp32 = head_state['weight'].detach().float().cpu()
        b_fp32 = head_state['bias'].detach().float().cpu()

        if masks_path is not None:
            masks      = torch.load(masks_path, map_location='cpu')
            head_masks = masks['head_masks']
            mlp_masks  = masks['mlp_masks']
            head_masks_dev = [m.to(device) for m in head_masks]
            mlp_masks_dev  = [m.to(device) for m in mlp_masks]
        else:
            head_masks_dev = None
            mlp_masks_dev  = None

        fp32_head = nn.Linear(w_fp32.shape[1], NUM_CLASSES, bias=True).to(device).eval()
        fp32_head.weight.data = w_fp32.to(device)
        fp32_head.bias.data   = b_fp32.to(device)

        correct_fp32 = total = 0
        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc=f"  FP32  {level_name}", leave=False):
                images, labels = images.to(device), labels.to(device)

                logits = masked_forward_fp32(
                        fp32_base, images, fp32_head,
                        head_masks_dev, mlp_masks_dev
                )

                _, predicted = logits.max(1)
                correct_fp32 += (predicted == labels).sum().item()
                total        += labels.size(0)

        acc_fp32 = 100. * correct_fp32 / total
        del fp32_head

        if masks_path is not None:
            head_masks_cpu = masks['head_masks']
            mlp_masks_cpu  = masks['mlp_masks']

        correct_quant = total = 0
        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc=f"  Quant {level_name}", leave=False):
                images_cpu = images.cpu()

                feats = masked_forward_quant(
                        quant_backbone, images_cpu,
                        head_masks_cpu, mlp_masks_cpu
                )

                logits   = apply_classifier_like_c_batch(feats, w_fp32, b_fp32)
                _, predicted = logits.max(1)
                correct_quant += (predicted == labels.cpu()).sum().item()
                total         += labels.size(0)

        acc_quant = 100. * correct_quant / total

        drop = acc_fp32 - acc_quant
        all_results[MODEL_NAME][level_name] = {
            'fp32':  acc_fp32,
            'quant': acc_quant,
            'drop':  drop,
        }

        print(f"   FP32: {acc_fp32:.2f}%  |  Quantized (INT8-RANGE): {acc_quant:.2f}%  |  Drop: {drop:+.2f}%")

        torch.cuda.empty_cache()

    del fp32_base
    torch.cuda.empty_cache()

print("\n\n" + "="*110)
print("SUMMARY: FP32 vs QUANTIZED ACCURACY")
print("="*110)

for MODEL_NAME, results in all_results.items():
    print(f"\n{MODEL_NAME}:")
    print(f"  {'Level':<30s} {'FP32':>10s} {'Quantized (INT8-RANGE)':>25s} {'Drop':>10s}")
    print(f"  {'-'*80}")
    for level_name, r in results.items():
        print(f"  {level_name:<30s} {r['fp32']:>9.2f}%  {r['quant']:>23.2f}%  {r['drop']:>+9.2f}%")

print("\n" + "="*110)
print("CELL Q3b COMPLETED (INT8-RANGE QUANTIZATION, PRECISION-MATCHED TO C CODE)")
print("="*110)